# Step 17: Voice vs face
The same XGBoost model with **voice only**, **face only** and **both**.
*Same as `src/17_ablation.py`.*

Go to the project folder.

In [ ]:
import os
if os.getcwd().endswith("notebooks"):
    os.chdir("..")          # work from the project folder

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

In [ ]:
def ccc(y_true, y_pred):
    # Concordance Correlation Coefficient: 1 = perfect agreement
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    covariance = np.mean((y_true - y_true.mean()) * (y_pred - y_pred.mean()))
    return 2 * covariance / (y_true.var() + y_pred.var() + (y_true.mean() - y_pred.mean()) ** 2 + 1e-9)

**Train on one part, test on another, using only the given columns.**

In [ ]:
def train_and_test(train, test, cols):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train[cols].fillna(0))
    X_test = scaler.transform(test[cols].fillna(0))
    model = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8,
                         colsample_bytree=0.8, tree_method="hist", random_state=42, n_jobs=-1)
    model.fit(X_train, train["engagement"])
    return ccc(test["engagement"], model.predict(X_test))

In [ ]:
df = pd.read_parquet("data/features/noxi_engagement.parquet")
bookkeeping = ["face_0", "face_1", "face_2", "face_3", "face_4"]
voice, face = [], []
for c in df.columns:
    if c.startswith("voice_"):
        voice.append(c)
    elif c.startswith("face_") and c not in bookkeeping:
        face.append(c)
inputs = {"voice": voice, "face": face, "both": voice + face}
languages = ["English", "French", "German"]
print(len(voice), "voice and", len(face), "face features")

In [ ]:
# the 3 sessions of each language
english = sorted(df[df["language"] == "English"]["session"].unique())
french = sorted(df[df["language"] == "French"]["session"].unique())
german = sorted(df[df["language"] == "German"]["session"].unique())

# fold 1 tests on the 1st session of each language, fold 2 on the 2nd, fold 3 on the 3rd
folds = []
for k in range(3):
    folds.append([english[k], french[k], german[k]])
print(folds)

**For each input:** unseen conversations (3 folds), within each language, and across languages.

In [ ]:
for name, cols in inputs.items():
    unseen = []
    for test_sessions in folds:
        test = df["session"].isin(test_sessions)
        unseen.append(train_and_test(df[~test], df[test], cols))

    within = []
    for l in languages:
        data = df[df["language"] == l]
        for s in data["session"].unique():
            within.append(train_and_test(data[data["session"] != s], data[data["session"] == s], cols))

    cross = []
    for a in languages:
        for b in languages:
            if a != b:
                cross.append(train_and_test(df[df["language"] == a], df[df["language"] == b], cols))

    drop = 100 * (np.mean(within) - np.mean(cross)) / np.mean(within)
    print(name.ljust(6), "unseen", round(np.mean(unseen), 3), "| within", round(np.mean(within), 3),
          "| cross", round(np.mean(cross), 3), "| drop", round(drop), "%")

**The result saved by the script**, for comparison.

In [ ]:
print(pd.read_csv("reports/ablation_results.csv")[["modality", "session_ccc", "within_ccc", "cross_ccc", "drop_pct"]].round(3))